In [1]:
# ==========================================================
# 1. IMPORTACIÓN DE LIBRERÍAS
# ==========================================================

import torch

import numpy as np
import pandas as pd

from matplotlib import pyplot as plt

from tqdm import tqdm

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


In [2]:
# ==========================================================
# 2. DETECTAR GPU
# ==========================================================

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Dispositivo utilizado:", device)

Dispositivo utilizado: cpu


In [3]:
# ==========================================================
# 3. CARGAR DATASET
# ==========================================================

data = pd.read_csv("CHARTEVENTS.csv")

print("Dataset cargado correctamente.")

print("Dimensiones del dataset:")
print(data.shape)

data.head()

C:\Users\VICTUS hp\AppData\Local\Temp\ipykernel_60076\2071379833.py:5: DtypeWarning: Columns (0: value, 1: valueuom, 2: resultstatus, 3: stopped) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv("CHARTEVENTS.csv")


Dataset cargado correctamente.
Dimensiones del dataset:
(758355, 15)


,row_id,subject_id,hadm_id,icustay_id,itemid,charttime,storetime,cgid,value,valuenum,valueuom,warning,error,resultstatus,stopped
0,5279021,40124,126179,279554.0,223761,2130-02-04 04:00:00,2130-02-04 04:35:00,19085,95.9,95.9,?F,0.0,0.0,NaN,NaN
1,5279022,40124,126179,279554.0,224695,2130-02-04 04:25:00,2130-02-04 05:55:00,18999,2222221.7,2222221.7,cmH2O,0.0,0.0,NaN,NaN
2,5279023,40124,126179,279554.0,220210,2130-02-04 04:30:00,2130-02-04 04:43:00,21452,15.0,15.0,insp/min,0.0,0.0,NaN,NaN
3,5279024,40124,126179,279554.0,220045,2130-02-04 04:32:00,2130-02-04 04:43:00,21452,94.0,94.0,bpm,0.0,0.0,NaN,NaN
4,5279025,40124,126179,279554.0,220179,2130-02-04 04:32:00,2130-02-04 04:43:00,21452,163.0,163.0,mmHg,0.0,0.0,NaN,NaN


In [4]:
# ==========================================================
# 4. EXPLORACIÓN INICIAL
# ==========================================================

print("\nColumnas del dataset:")
print(list(data.columns))

print("\nValores nulos por columna:")
print(data.isnull().sum())

print("\nTipos de datos:")
print(data.dtypes)


Columnas del dataset:
['row_id', 'subject_id', 'hadm_id', 'icustay_id', 'itemid', 'charttime', 'storetime', 'cgid', 'value', 'valuenum', 'valueuom', 'warning', 'error', 'resultstatus', 'stopped']

Valores nulos por columna:
row_id               0
subject_id           0
hadm_id              0
icustay_id          81
itemid               0
charttime            0
storetime            0
cgid                 0
value            32848
valuenum        434471
valueuom        518500
warning         376076
error           376076
resultstatus    736681
stopped         383706
dtype: int64

Tipos de datos:
row_id            int64
subject_id        int64
hadm_id           int64
icustay_id      float64
itemid            int64
charttime           str
storetime           str
cgid              int64
value            object
valuenum        float64
valueuom            str
warning         float64
error           float64
resultstatus        str
stopped             str
dtype: object


In [7]:
# ==========================================================
# 5. SELECCIÓN DE COLUMNAS IMPORTANTES
# ==========================================================

features = [
    "subject_id",   # ID del paciente
    "itemid",       # tipo de medición médica
    "valuenum"      # valor numérico de la medición
]

target = "valuenum"

df = data[features].copy()

print("Columnas seleccionadas correctamente:")
print(df.columns)

print("Dimensiones del dataframe:")
print(df.shape)

df.head()

Columnas seleccionadas correctamente:
Index(['subject_id', 'itemid', 'valuenum'], dtype='str')
Dimensiones del dataframe:
(758355, 3)


,subject_id,itemid,valuenum
0,40124,223761,95.9
1,40124,224695,2222221.7
2,40124,220210,15.0
3,40124,220045,94.0
4,40124,220179,163.0


In [8]:
# ==========================================================
# 6. LIMPIEZA DE DATOS
# ==========================================================

# eliminar filas donde no existe el valor numérico
df = df.dropna(subset=["valuenum"])

print("Filas después de eliminar nulos:", len(df))

print("\nNulos restantes:")
print(df.isnull().sum())

Filas después de eliminar nulos: 323884

Nulos restantes:
subject_id    0
itemid        0
valuenum      0
dtype: int64


In [9]:
# ==========================================================
# 7. IMPUTAR VALORES NULOS
# ==========================================================

for col in df.columns:

    if df[col].dtype == "object":

        df[col] = df[col].fillna(df[col].mode()[0])

    else:

        df[col] = df[col].fillna(df[col].mean())

print("Nulos imputados correctamente")

print("\nNulos restantes:")
print(df.isnull().sum())

Nulos imputados correctamente

Nulos restantes:
subject_id    0
itemid        0
valuenum      0
dtype: int64


In [10]:
# ==========================================================
# 8. PREPARAR MATRICES X e y
# ==========================================================

X = df[features].values
y = df[target].values

print("Forma de X:", X.shape)
print("Forma de y:", y.shape)

Forma de X: (323884, 3)
Forma de y: (323884,)
